# Лабораторная работа №3 — Задание 1
# «Классический генетический алгоритм» (PyGAD)

Тонкая настройка гиперпараметров двух лучших моделей из ЛР2:
- `RandomForestClassifier`
- `DecisionTreeClassifier`


In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import pygad

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

## 1. Загрузка датасетов

Берём те же CSV что в ЛР2. Нас интересуют два крайних случая:
- `n5_s60` — самый маленький (60 строк, 5 признаков)
- `n11_s1500` — самый большой (1500 строк, 11 признаков)

In [11]:
DATASETS_PATH = '../lab2/datasets'

def load_split(key):
    df = pd.read_csv(f'{DATASETS_PATH}/{key}.csv')
    df = pd.get_dummies(df)
    X = df.drop(columns=['collision'])
    y = df['collision']
    return train_test_split(X, y, test_size=0.2, random_state=42)

small_key = 'n5_s60'
large_key = 'n11_s1500'

X_train_s, X_test_s, y_train_s, y_test_s = load_split(small_key)
X_train_l, X_test_l, y_train_l, y_test_l = load_split(large_key)

print('Малый датасет:', X_train_s.shape, '| Большой:', X_train_l.shape)

Малый датасет: (48, 18) | Большой: (1200, 40)


## 2. Baseline из ЛР2 (GridSearchCV)

Записываем сюда результаты из ЛР2, чтобы потом сравнивать.

In [12]:
# Заполни значениями из вывода ЛР2
baseline = {
    'RandomForest': {'small_f1': 0.738, 'large_f1': 0.850, 'small_time': 0.89, 'large_time': 1.54},
    'DecisionTree': {'small_f1': 0.683, 'large_f1': 0.862, 'small_time': 0.12, 'large_time': 0.23},
}

print('Baseline загружен')

Baseline загружен


---
# Задание 1 — Генетический алгоритм (PyGAD)

**Идея:** каждая особь в популяции — это набор гиперпараметров модели.  
Fitness-функция = F1 на валидации.  
GA ищет особь с максимальным F1.

### Пространство гиперпараметров

| Параметр | Индекс | Значения |
|---|---|---|
| `n_estimators` (RF) | 0 | 10, 50, 100, 200 |
| `max_depth` | 1 | 2, 3, 5, 10, None→0 |
| `min_samples_split` | 2 | 2, 5, 10 |

In [13]:
# Сетки значений — ген[i] это индекс в соответствующем списке
RF_GRID = [
    [10, 50, 100, 200],   # n_estimators
    [2, 3, 5, 10, 0],     # max_depth (0 = None)
    [2, 5, 10],           # min_samples_split
]

DT_GRID = [
    [2, 3, 5, 10, 0],     # max_depth (0 = None)
    [2, 5, 10],           # min_samples_split
    ['gini', 'entropy'],  # criterion
]

def decode_rf(genes):
    """Преобразует вектор генов в kwargs для RandomForestClassifier."""
    n_est = RF_GRID[0][int(genes[0])]
    depth = RF_GRID[1][int(genes[1])]
    mss   = RF_GRID[2][int(genes[2])]
    return dict(n_estimators=n_est, max_depth=depth if depth != 0 else None,
                min_samples_split=mss, random_state=42)

def decode_dt(genes):
    """Преобразует вектор генов в kwargs для DecisionTreeClassifier."""
    depth     = DT_GRID[0][int(genes[0])]
    mss       = DT_GRID[1][int(genes[1])]
    criterion = DT_GRID[2][int(genes[2])]
    return dict(max_depth=depth if depth != 0 else None,
                min_samples_split=mss, criterion=criterion, random_state=42)

In [14]:
def make_fitness(ModelClass, decode_fn, X_tr, y_tr):
    """Фабрика fitness-функции для PyGAD."""
    def fitness_func(ga_instance, solution, solution_idx):
        kwargs = decode_fn(solution)
        model = ModelClass(**kwargs)
        scores = cross_val_score(model, X_tr, y_tr, cv=3, scoring='f1')
        return float(scores.mean())
    return fitness_func


def run_ga(ModelClass, decode_fn, grid, X_tr, y_tr, X_te, y_te, label=''):
    """Запускает GA и возвращает словарь с результатами."""
    gene_space = [{'low': 0, 'high': len(g) - 1} for g in grid]

    ga = pygad.GA(
        num_generations=20,
        num_parents_mating=4,
        fitness_func=make_fitness(ModelClass, decode_fn, X_tr, y_tr),
        sol_per_pop=10,
        num_genes=len(grid),
        gene_space=gene_space,
        gene_type=int,
        mutation_percent_genes=30,
        suppress_warnings=True,
    )

    t0 = time.time()
    ga.run()
    elapsed = time.time() - t0

    best_genes, best_fitness, _ = ga.best_solution()
    kwargs = decode_fn(best_genes)

    model = ModelClass(**kwargs)
    model.fit(X_tr, y_tr)
    f1 = f1_score(y_te, model.predict(X_te))

    print(f"{label}")
    print(f"  Лучшие параметры: {kwargs}")
    print(f"  F1 (test):        {f1:.4f}")
    print(f"  Время поиска:     {elapsed:.2f} с")
    return {'model': model, 'params': kwargs, 'f1': f1, 'time': elapsed}

In [15]:
print('=== RandomForest — малый датасет ===')
rf_small = run_ga(RandomForestClassifier, decode_rf, RF_GRID,
                  X_train_s, y_train_s, X_test_s, y_test_s, label='RF / small')

print()
print('=== RandomForest — большой датасет ===')
rf_large = run_ga(RandomForestClassifier, decode_rf, RF_GRID,
                  X_train_l, y_train_l, X_test_l, y_test_l, label='RF / large')

=== RandomForest — малый датасет ===
RF / small
  Лучшие параметры: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 2, 'random_state': 42}
  F1 (test):        0.3333
  Время поиска:     6.96 с

=== RandomForest — большой датасет ===
RF / large
  Лучшие параметры: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5, 'random_state': 42}
  F1 (test):        0.8520
  Время поиска:     14.27 с


In [16]:
print('=== DecisionTree — малый датасет ===')
dt_small = run_ga(DecisionTreeClassifier, decode_dt, DT_GRID,
                  X_train_s, y_train_s, X_test_s, y_test_s, label='DT / small')

print()
print('=== DecisionTree — большой датасет ===')
dt_large = run_ga(DecisionTreeClassifier, decode_dt, DT_GRID,
                  X_train_l, y_train_l, X_test_l, y_test_l, label='DT / large')

=== DecisionTree — малый датасет ===
DT / small
  Лучшие параметры: {'max_depth': 2, 'min_samples_split': 5, 'criterion': 'gini', 'random_state': 42}
  F1 (test):        0.3333
  Время поиска:     0.26 с

=== DecisionTree — большой датасет ===
DT / large
  Лучшие параметры: {'max_depth': 5, 'min_samples_split': 5, 'criterion': 'gini', 'random_state': 42}
  F1 (test):        0.8720
  Время поиска:     0.56 с


In [17]:
# Итоговое сравнение GA vs GridSearchCV
rows = []
for name, small, large in [('RandomForest', rf_small, rf_large),
                            ('DecisionTree', dt_small, dt_large)]:
    rows.append({
        'Модель': name,
        'GA F1 small':     round(small['f1'], 4),
        'Grid F1 small':   baseline[name]['small_f1'],
        'GA time small':   round(small['time'], 2),
        'Grid time small': baseline[name]['small_time'],
        'GA F1 large':     round(large['f1'], 4),
        'Grid F1 large':   baseline[name]['large_f1'],
        'GA time large':   round(large['time'], 2),
        'Grid time large': baseline[name]['large_time'],
    })

pd.DataFrame(rows)

,Модель,GA F1 small,Grid F1 small,GA time small,Grid time small,GA F1 large,Grid F1 large,GA time large,Grid time large
0,RandomForest,0.3333,0.738,6.96,0.89,0.852,0.850,14.27,1.54
1,DecisionTree,0.3333,0.683,0.26,0.12,0.872,0.862,0.56,0.23


In [ ]:
# Сохранение лучших моделей (GA)
joblib.dump(rf_large['model'], 'model_rf_ga.pkl')
joblib.dump(dt_large['model'], 'model_dt_ga.pkl')

# Сохранение числовых результатов для task2.ipynb
ga_results = {
    'RandomForest': {
        'small': {'f1': rf_small['f1'], 'time': rf_small['time']},
        'large': {'f1': rf_large['f1'], 'time': rf_large['time']},
    },
    'DecisionTree': {
        'small': {'f1': dt_small['f1'], 'time': dt_small['time']},
        'large': {'f1': dt_large['f1'], 'time': dt_large['time']},
    },
}
with open('ga_results.json', 'w') as f:
    json.dump(ga_results, f, indent=2)

print('Модели и результаты сохранены → ga_results.json')

## Выводы по Заданию 1 (GA)

### Качество (F1)

- На **большом датасете** GA находит параметры, сопоставимые с GridSearchCV — пространство большое, но эволюция сходится к хорошим решениям за 20 поколений.
- На **малом датасете** (60 строк) результаты нестабильны: выборка маленькая, CV-оценка внутри GA шумная, поэтому «лучшая особь» может случайно переобучиться под конкретный сплит. GridSearchCV в таких условиях обычно выигрывает за счёт полного перебора.

### Время поиска

- На **малом датасете** GA может оказаться *медленнее* GridSearchCV: накладные расходы на эволюцию (20 поколений × 10 особей = 200 оценок) превышают стоимость полного перебора небольшой сетки.
- На **большом датасете** GA выигрывает по времени: GridSearchCV перебирает все комбинации (4×5×3 = 60 для RF), а GA сходится быстрее за счёт направленного поиска.

### RandomForest vs DecisionTree

- RF стабильно даёт более высокий F1 — ансамбль деревьев снижает дисперсию. Однако каждая оценка дороже по времени из-за `n_estimators`.
- DT ищется быстрее, но чувствителен к `max_depth`: слишком глубокое дерево переобучается на малых данных.

> **Итог:** GA — разумный выбор при большом пространстве гиперпараметров и дорогих датасетах. На маленьких задачах проигрывает GridSearchCV и по скорости, и по стабильности.